# Aadhaar KYC Face Matching Pipeline - SageMaker

Exact same pipeline as local. Requires GPU instance: `ml.g5.xlarge` (A10G 24GB) recommended.

Volume size: **100GB** (models are ~12GB total)

## Step 1: Clone repo

In [ ]:
!git clone https://github.com/MlvPrasadOfficial/GENAI_AADHAR.git
%cd GENAI_AADHAR

## Step 2: Check GPU

In [ ]:
!nvidia-smi

## Step 3: Install Python dependencies

In [ ]:
# Install order matters — do NOT rearrange
!pip install torch==2.3.1+cu121 torchvision==0.18.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4
!pip install onnxruntime-gpu==1.19.2
!pip install basicsr==1.4.2
!pip install realesrgan==0.3.0
!pip install insightface==0.7.3
!pip install opencv-python==4.10.0.84 Pillow==10.4.0 PyYAML==6.0.2 requests==2.32.3 tqdm==4.66.4 PyMuPDF==1.24.5

In [ ]:
# Verify GPU
import torch
print(f"PyTorch CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

import onnxruntime
print(f"ONNX Runtime: {onnxruntime.get_available_providers()}")

## Step 4: Install Ollama + Load Qwen2.5-VL from local HuggingFace download

In [ ]:
# Install Ollama on Linux
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start Ollama server in background
import subprocess, time

proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print(f"Ollama server started (PID: {proc.pid})")

# Verify it's running
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    print(f"Ollama is running: {r.status_code}")
except:
    print("ERROR: Ollama not responding. Check logs.")

In [ ]:
# Load model from local HuggingFace GGUF file
# UPDATE THIS PATH to where your team downloaded the model
import os

GGUF_PATH = "/home/ec2-user/models/qwen2.5-vl-7b-instruct.gguf"  # <-- UPDATE THIS

if not os.path.exists(GGUF_PATH):
    raise FileNotFoundError(
        f"Model not found at {GGUF_PATH}\n"
        f"Download from HuggingFace first and update GGUF_PATH above."
    )

# Create Modelfile and register with Ollama
modelfile = f"FROM {GGUF_PATH}\n"
with open("/tmp/Modelfile", "w") as f:
    f.write(modelfile)

!ollama create qwen2.5vl:7b -f /tmp/Modelfile
print("Loaded Qwen2.5-VL from local GGUF")

In [ ]:
# Verify model is available
!ollama list

## Step 5: Setup pipeline model weights (from local HuggingFace downloads)

In [ ]:
# Link pipeline models from local HuggingFace downloads
# UPDATE these paths to where your team has the models

!mkdir -p models/insightface/models models/realesrgan

# InsightFace buffalo_l (5 ONNX files)
!ln -s /home/ec2-user/huggingface-models/buffalo_l models/insightface/models/buffalo_l

# Real-ESRGAN
!cp /home/ec2-user/huggingface-models/RealESRGAN_x4plus.pth models/realesrgan/

In [ ]:
# Verify models are in place
!echo "=== InsightFace ==="  && ls models/insightface/models/buffalo_l/
!echo "=== Real-ESRGAN ===" && ls models/realesrgan/

## Step 6: Upload test images

Upload your Aadhaar cards and selfies via Jupyter file browser, or copy from S3.

In [ ]:
!mkdir -p FILES/AADHAR FILES/SELFIE

# From S3:
# !aws s3 cp s3://your-bucket/test-data/AADHAR/ FILES/AADHAR/ --recursive
# !aws s3 cp s3://your-bucket/test-data/SELFIE/ FILES/SELFIE/ --recursive

!echo "Aadhaar files:" && ls FILES/AADHAR/
!echo "Selfie files:"  && ls FILES/SELFIE/

## Step 7: Run tests

In [ ]:
!pip install pytest==8.3.2
!python -m pytest tests/ -v -m "not integration" --tb=short 2>&1 | tail -5

## Step 8: Run pipeline

In [ ]:
# Single pair
!python main.py --aadhaar FILES/AADHAR/AADHAR001.jpg --selfie FILES/SELFIE/USER_01.jpg --verbose

In [ ]:
# Batch mode - all 90 pairs
!python main.py --batch pairs.csv --verbose

In [ ]:
# Check results
import os
latest = sorted([d for d in os.listdir('logs') if d.startswith('batch_')])[-1]
!cat logs/{latest}/summary.txt

## Step 9: Use as Python library

In [ ]:
from utils.config_loader import load_config
from pipeline.orchestrator import KYCPipelineOrchestrator

config = load_config('config.yaml')
pipeline = KYCPipelineOrchestrator(config)
pipeline.load()

aadhaar_bytes = open('FILES/AADHAR/AADHAR001.jpg', 'rb').read()
selfie_bytes  = open('FILES/SELFIE/USER_01.jpg', 'rb').read()

result = pipeline.run(aadhaar_bytes, selfie_bytes)

print(f"Match:      {result.match}")
print(f"Confidence: {result.confidence_pct}%")
print(f"Cosine:     {result.cosine_score:.4f}")
print(f"Fused:      {result.fused_score:.4f}")
print(f"VLM:        {result.vlm_reasoning}")
print(f"Aadhaar:    {result.aadhaar_gender} age {result.aadhaar_age}")
print(f"Selfie:     {result.selfie_gender} age {result.selfie_age}")